# DinoV2 Dogs-vs-Cats Hypothesis: Compare Uploaded Results (No Retraining)

This notebook is a consumer workflow: it compares previously exported experiment results from a Kaggle input dataset.

Default behavior: no training. Optional fallback can run evaluate-only from uploaded checkpoints when result files are missing.

Flow: bootstrap -> load bundle index -> validate/uploaded results -> optional fallback evaluate -> compare -> review artifacts.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

REPO_URL = 'https://github.com/mruniverse8/kaggle-experiments-.git'
REPO_DIR = Path('/kaggle/working/kaggle-experiments-')
BRANCH = 'dogs_vs_cats_v2'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', '--all'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('Git branch:', current_branch)
print('Git commit:', current_commit)
print('Repo ready at:', REPO_DIR)


In [ ]:
import os
import json
from pathlib import Path

PATHS_CFG = os.environ.get('PATHS_CFG', 'dogs_vs_cats/configs/paths_kaggle.json')
ARTIFACT_INPUT_ROOT = os.environ.get('ARTIFACT_INPUT_ROOT', '/kaggle/input/dogs-vs-cats-artifacts')
ENABLE_FALLBACK_EVAL = os.environ.get('ENABLE_FALLBACK_EVAL', '0') == '1'
FALLBACK_SUMMARY_SUFFIX = os.environ.get('FALLBACK_SUMMARY_SUFFIX', 'eval_only')
SELECT_EXPERIMENTS = [x.strip() for x in os.environ.get('SELECT_EXPERIMENTS', '').split(',') if x.strip()]
MAX_EXPERIMENTS = int(os.environ.get('MAX_EXPERIMENTS', '2'))

PATHS = json.loads(Path(PATHS_CFG).read_text())
artifact_root = Path(ARTIFACT_INPUT_ROOT)
bundle_index_path = artifact_root / 'bundle_index.json'
if not bundle_index_path.exists():
    raise FileNotFoundError(f'Bundle index not found: {bundle_index_path}')

bundle_index = json.loads(bundle_index_path.read_text())
entries = bundle_index.get('experiments', [])
if not entries:
    raise RuntimeError(f'No experiments in bundle index: {bundle_index_path}')

if SELECT_EXPERIMENTS:
    wanted = set(SELECT_EXPERIMENTS)
    selected_entries = [e for e in entries if e.get('experiment_name') in wanted]
    found = {e.get('experiment_name') for e in selected_entries}
    missing = sorted(wanted - found)
    if missing:
        raise RuntimeError(f'SELECT_EXPERIMENTS not found in bundle index: {missing}')
else:
    selected_entries = entries[:max(1, MAX_EXPERIMENTS)]

print('Using paths config:', PATHS_CFG)
print('Artifact input root:', artifact_root)
print('Enable fallback evaluate:', ENABLE_FALLBACK_EVAL)
print('Fallback summary suffix:', FALLBACK_SUMMARY_SUFFIX)
print('Selected experiments:')
for row in selected_entries:
    print('-', row.get('experiment_name'), '->', row.get('manifest_path'))


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

RESULT_REQUIRED_KEYS = ['metrics_csv', 'val_predictions_csv', 'test_predictions_csv', 'submission_csv']

def infer_profile(experiment_name: str) -> str:
    lower = experiment_name.lower()
    if 'no_aug' in lower:
        return 'no_aug'
    if 'strong_aug' in lower:
        return 'strong_aug'
    return 'unknown'

def manifest_files_abs(manifest_path: Path, manifest: dict) -> dict:
    exp_root = manifest_path.parent
    files_rel = manifest.get('files', {})
    out = {}
    for key, rel_path in files_rel.items():
        out[key] = str((exp_root / rel_path).resolve())
    return out

def missing_result_keys(files_abs: dict) -> list:
    missing = []
    for key in RESULT_REQUIRED_KEYS:
        path = files_abs.get(key, '')
        if not path or not Path(path).exists():
            missing.append(key)
    return missing

RUNS = []
for entry in selected_entries:
    manifest_rel = entry.get('manifest_path', '')
    manifest_path = artifact_root / manifest_rel
    if not manifest_path.exists():
        raise FileNotFoundError(f'Manifest not found for entry {entry}: {manifest_path}')

    manifest = json.loads(manifest_path.read_text())
    exp_name = manifest.get('experiment_name', entry.get('experiment_name', 'unknown_experiment'))
    files_abs = manifest_files_abs(manifest_path, manifest)
    missing_keys = missing_result_keys(files_abs)

    if missing_keys and not ENABLE_FALLBACK_EVAL:
        raise RuntimeError(
            f'Missing required result artifacts for {exp_name}: {missing_keys}. ' 
            'Set ENABLE_FALLBACK_EVAL=1 to run evaluate-only fallback from checkpoint.'
        )

    if missing_keys and ENABLE_FALLBACK_EVAL:
        exp_cfg_rel = manifest.get('files', {}).get('experiment_config', '')
        best_ckpt_rel = manifest.get('files', {}).get('best_checkpoint', '')
        if not exp_cfg_rel or not best_ckpt_rel:
            raise RuntimeError(f'Cannot fallback-evaluate {exp_name}: missing experiment_config/best_checkpoint in manifest')

        exp_cfg_path = manifest_path.parent / exp_cfg_rel
        best_ckpt_path = manifest_path.parent / best_ckpt_rel
        if not exp_cfg_path.exists():
            raise FileNotFoundError(f'Fallback experiment config missing: {exp_cfg_path}')
        if not best_ckpt_path.exists():
            raise FileNotFoundError(f'Fallback checkpoint missing: {best_ckpt_path}')

        exp_cfg_payload = json.loads(exp_cfg_path.read_text())
        source_exp_name = str(exp_cfg_payload.get('experiment_name', exp_name))
        suffix = str(FALLBACK_SUMMARY_SUFFIX).strip()
        output_exp_name = f'{source_exp_name}_{suffix}' if suffix else source_exp_name

        print(f'[fallback-evaluate] {exp_name} -> rebuilding missing results {missing_keys}')
        subprocess.run([
            sys.executable,
            'dogs_vs_cats/src/dinov2_pipeline.py',
            '--mode', 'evaluate',
            '--paths-config', PATHS_CFG,
            '--experiment-config', str(exp_cfg_path),
            '--checkpoint-path', str(best_ckpt_path),
            '--summary-suffix', FALLBACK_SUMMARY_SUFFIX,
        ], check=True)

        eval_summary_path = Path(PATHS['reports_dir']) / f'{output_exp_name}_training_summary.json'
        if not eval_summary_path.exists():
            raise FileNotFoundError(f'Fallback summary not found: {eval_summary_path}')

        eval_summary = json.loads(eval_summary_path.read_text())
        run_files = {k: str(v) for k, v in eval_summary.get('files', {}).items()}
        run_files['training_summary_json'] = str(eval_summary_path)
        run_files['best_checkpoint'] = str(eval_summary.get('best_checkpoint', ''))

        RUNS.append({
            'profile': infer_profile(exp_name),
            'experiment_name': eval_summary.get('experiment_name', exp_name),
            'source': 'fallback_evaluate',
            'final_val_metrics': eval_summary.get('final_val_metrics', {}),
            'monitor': eval_summary.get('monitor', ''),
            'best_monitor': eval_summary.get('best_monitor'),
            'best_epoch': eval_summary.get('best_epoch'),
            'files': run_files,
        })
        continue

    training_summary = {}
    training_summary_path = files_abs.get('training_summary_json', '')
    if training_summary_path and Path(training_summary_path).exists():
        training_summary = json.loads(Path(training_summary_path).read_text())

    summary_block = manifest.get('training_summary', {})
    RUNS.append({
        'profile': infer_profile(exp_name),
        'experiment_name': exp_name,
        'source': 'uploaded_results',
        'final_val_metrics': manifest.get('final_val_metrics', {}),
        'monitor': summary_block.get('monitor', ''),
        'best_monitor': summary_block.get('best_monitor'),
        'best_epoch': summary_block.get('best_epoch'),
        'files': files_abs,
        'training_summary': training_summary,
    })

print('Prepared runs:')
for run in RUNS:
    print('-', run['experiment_name'], '| source:', run['source'])


In [ ]:
import pandas as pd

rows = []
for run in RUNS:
    metrics = run.get('final_val_metrics', {})
    rows.append({
        'profile': run.get('profile', ''),
        'experiment_name': run.get('experiment_name', ''),
        'source': run.get('source', ''),
        'val_auc': metrics.get('val_auc'),
        'val_logloss': metrics.get('val_logloss'),
        'val_accuracy': metrics.get('val_accuracy'),
        'val_loss': metrics.get('val_loss'),
        'monitor': run.get('monitor', ''),
        'best_monitor': run.get('best_monitor'),
        'best_epoch': run.get('best_epoch'),
    })

compare_df = pd.DataFrame(rows).sort_values(by=['val_auc', 'val_accuracy'], ascending=[False, False]).reset_index(drop=True)
print('Validation comparison:')
display(compare_df)

base_row = compare_df.loc[compare_df['profile'] == 'no_aug']
aug_row = compare_df.loc[compare_df['profile'] == 'strong_aug']
if len(base_row) == 1 and len(aug_row) == 1:
    base_row = base_row.iloc[0]
    aug_row = aug_row.iloc[0]
    deltas = {
        'delta_val_auc': float(aug_row['val_auc'] - base_row['val_auc']),
        'delta_val_logloss': float(aug_row['val_logloss'] - base_row['val_logloss']),
        'delta_val_accuracy': float(aug_row['val_accuracy'] - base_row['val_accuracy']),
    }
    print('Strong_aug - no_aug deltas:', deltas)
else:
    print('Could not compute no_aug vs strong_aug deltas (profiles not uniquely available).')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_keys = [
    'train_loss_plot',
    'val_metrics_plot',
    'grad_norm_plot',
    'val_confusion_matrix_plot',
]

for run in RUNS:
    print('\nPLOTS:', run['experiment_name'])
    files = run.get('files', {})
    for key in plot_keys:
        path = files.get(key, "")
        if not path:
            continue
        p = Path(path)
        if not p.exists():
            continue
        plt.figure(figsize=(8, 4))
        plt.imshow(mpimg.imread(p))
        plt.title(f"{run['source']} | {p.name}")
        plt.axis('off')
        plt.show()


In [ ]:
from pathlib import Path

for run in RUNS:
    print('\n' + '=' * 80)
    print('EXPERIMENT:', run.get('experiment_name', ''))
    print('SOURCE:', run.get('source', ''))
    print('MONITOR:', run.get('monitor', ''))
    print('BEST MONITOR:', run.get('best_monitor'))
    print('BEST EPOCH:', run.get('best_epoch'))
    files = run.get('files', {})
    for key in sorted(files.keys()):
        path = Path(files[key])
        print('-', key, '->', path, '| exists=', path.exists())
